In [1]:
import pandas as pd
import sqlite3
import io
from datetime import datetime

# ==============================================================================
# 1. CARGA Y GENERACIÓN DE DATASETS ORIGINALES (CSV Sucios)
# ==============================================================================
miembros_raw = """id_miembro,miembro_info,email,peso_kg
1,Maria Juliana Herrera - Activo,maju@correo.com,60.5
2,Mariana Carmona - Inactivo,,55.0
3,Andres perez - Activo,aperez@gym.com,
4,LUISA GOMEZ - Activo,luisa@gym.com,62.0"""

entrenadores_raw = """id_entrenador,entrenador_info,turno
10,Carlos Ruiz - Crossfit,Mañana
11,Ana Lopez - Pesas,Tarde"""

planes_raw = """id_plan,plan_info,mensualidad,id_entrenador
101,Plan VIP - Acceso Total,120000,10
102,Plan Basico - Solo Mañanas,80000,10
103,Plan Musculacion - Pesas,95000,11"""

# Generamos los archivos físicos sucios
with open('miembros_sucios.csv', 'w') as f: f.write(miembros_raw)
with open('entrenadores_sucios.csv', 'w') as f: f.write(entrenadores_raw)
with open('planes_sucios.csv', 'w') as f: f.write(planes_raw)

# ==============================================================================
# 2. PANDAS Y NORMALIZACIÓN (1NF)
# ==============================================================================
# Miembros
df_miem = pd.read_csv(io.StringIO(miembros_raw))
df_miem[['nombre', 'estado']] = df_miem['miembro_info'].str.split(' - ', expand=True)
df_miem['nombre'] = df_miem['nombre'].str.title()
df_miem['email'] = df_miem['email'].fillna('no_registrado@fitlife.com')
df_miem['peso_kg'] = df_miem['peso_kg'].fillna(0.0) # Imputación de nulos
df_miem = df_miem.drop(columns=['miembro_info'])

# Entrenadores
df_ent = pd.read_csv(io.StringIO(entrenadores_raw))
df_ent[['nombre', 'especialidad']] = df_ent['entrenador_info'].str.split(' - ', expand=True)
df_ent = df_ent.drop(columns=['entrenador_info'])

# Planes
df_pla = pd.read_csv(io.StringIO(planes_raw))
df_pla[['nombre_plan', 'beneficio']] = df_pla['plan_info'].str.split(' - ', expand=True)
df_pla = df_pla.drop(columns=['plan_info'])

# Exportar limpios
df_miem.to_csv('miembros_limpios.csv', index=False)
df_ent.to_csv('entrenadores_limpios.csv', index=False)
df_pla.to_csv('planes_limpios.csv', index=False)

# ==============================================================================
# 3. MIGRACIÓN SQLITE Y ARQUITECTURA
# ==============================================================================
conn = sqlite3.connect('parcial_fitlife.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

df_miem.to_sql('miembros', conn, if_exists='replace', index=False)
df_ent.to_sql('entrenadores', conn, if_exists='replace', index=False)
df_pla.to_sql('planes', conn, if_exists='replace', index=False)

# Relación N:M (Suscripciones equivale a Ventas)
cursor.execute('''
CREATE TABLE IF NOT EXISTS suscripciones (
    id_suscripcion INTEGER PRIMARY KEY AUTOINCREMENT,
    id_miembro INTEGER,
    id_plan INTEGER,
    total_pagado REAL,
    fecha_inicio TEXT,
    FOREIGN KEY (id_miembro) REFERENCES miembros(id_miembro),
    FOREIGN KEY (id_plan) REFERENCES planes(id_plan)
)''')
conn.commit()

# ==============================================================================
# 4. BONO: POO Y CRUD
# ==============================================================================
class GestionGym:
    def __init__(self, db): self.db = db

    def vender_membresia(self, id_mie, id_pla):
        with sqlite3.connect(self.db) as c:
            precio = c.execute("SELECT mensualidad FROM planes WHERE id_plan = ?", (id_pla,)).fetchone()[0]
            fecha = datetime.now().strftime('%Y-%m-%d')
            c.execute("INSERT INTO suscripciones (id_miembro, id_plan, total_pagado, fecha_inicio) VALUES (?,?,?,?)",
                           (id_mie, id_pla, precio, fecha))
            print(f"✅ [CREATE] Suscripción registrada.")

    def ver_reporte_activos(self):
        with sqlite3.connect(self.db) as c:
            query = '''
            SELECT s.id_suscripcion, m.nombre, p.nombre_plan, e.nombre AS entrenador, s.total_pagado
            FROM suscripciones s
            JOIN miembros m ON s.id_miembro = m.id_miembro
            JOIN planes p ON s.id_plan = p.id_plan
            JOIN entrenadores e ON p.id_entrenador = e.id_entrenador
            '''
            print("\n--- REPORTE DE SUSCRIPCIONES (JOIN) ---")
            print(pd.read_sql(query, c))

    def actualizar_peso(self, id_mie, nuevo_peso):
        with sqlite3.connect(self.db) as c:
            c.execute("UPDATE miembros SET peso_kg = ? WHERE id_miembro = ?", (nuevo_peso, id_mie))
            print("✅ [UPDATE] Peso actualizado.")

    def cancelar_suscripcion(self, id_sus):
        with sqlite3.connect(self.db) as c:
            c.execute("DELETE FROM suscripciones WHERE id_suscripcion = ?", (id_sus,))
            print("✅ [DELETE] Suscripción cancelada.")

# Pruebas
app = GestionGym('parcial_fitlife.db')
app.vender_membresia(1, 101)
app.vender_membresia(2, 103)
app.vender_membresia(4, 102)
app.ver_reporte_activos()
app.actualizar_peso(3, 75.5)
app.cancelar_suscripcion(2)
conn.close()

✅ [CREATE] Suscripción registrada.
✅ [CREATE] Suscripción registrada.
✅ [CREATE] Suscripción registrada.

--- REPORTE DE SUSCRIPCIONES (JOIN) ---
   id_suscripcion                 nombre       nombre_plan   entrenador  \
0               1  Maria Juliana Herrera          Plan VIP  Carlos Ruiz   
1               2        Mariana Carmona  Plan Musculacion    Ana Lopez   
2               3            Luisa Gomez       Plan Basico  Carlos Ruiz   

   total_pagado  
0      120000.0  
1       95000.0  
2       80000.0  
✅ [UPDATE] Peso actualizado.
✅ [DELETE] Suscripción cancelada.
